In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

ModuleNotFoundError: No module named 'kagglehub'

### Annotation — Load the Titanic data
Loads the CSV into `df`. This is the raw working DataFrame used for initial inspection and cleaning. The Kaggle input path is environment-specific; it will need changing outside that environment.

In [ ]:
df = pd.read_csv('/kaggle/input/datasets/krpraveen0/ai-coding-class-titanic-dataset/Titanic.csv')

In [ ]:
df.head()

In [ ]:
df.info()

### Annotation — Normalize missing-value markers
The source dataset uses `?` as a missing-value marker in some fields. Replacing it with `np.nan` allows pandas missing-value operations such as `isna()` and scikit-learn imputers to recognize those entries.

In [ ]:
df = df.replace('?', np.nan)

In [ ]:
df.info()

### Annotation — Convert age to numeric
`errors='coerce'` converts parseable values to numbers and turns invalid strings into `NaN`. This makes malformed values explicit missing data rather than leaving the column as text.

In [ ]:
df['age'] = pd.to_numeric(df['age'], errors = 'coerce')

In [ ]:
df.info()

### Annotation — Convert selected columns to numeric
The loop applies numeric conversion to the listed columns. Verify each column's intended type before doing this: identifiers and categorical codes should not automatically be treated as continuous numeric measurements merely because they contain digits.

In [ ]:
column_list = ['age', 'sibsp', 'parch', 'fare', 'survived', 'pclass', 'body']

for col in column_list:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [ ]:
df.info()

### Annotation — Quantify missingness
Counts and percentages show how much data is missing in each column. This helps decide whether to impute, drop a feature, or investigate why values are absent. Missingness can itself carry information, so dropping a column should be justified.

In [ ]:
null_values = df.isna().sum().sort_values(ascending=False)
null_pct = (null_values/len(df)*100)
null_df = pd.DataFrame({'Null values':null_values, 'Null Percentage':null_pct})
print(null_df)

### Annotation — Drop high-missingness columns
Creates `df1` without `body` and `cabin`. This is a feature-selection decision, not a general rule. `body` is often recorded only for some deceased passengers, and `cabin` may be missing for many passengers; assess missingness and potential target leakage before deciding what to retain.

In [ ]:
df1 = df.drop(['body','cabin'], axis = 1)

In [ ]:
df1.head()

In [ ]:
df1.info()

### Annotation — Compare survival rates by boat availability
This compares target averages for rows with and without a recorded boat number. A boat number is closely tied to evacuation/survival outcomes and is not appropriate as a predictive feature if it would be unavailable at prediction time. Treat this as exploratory analysis only.

In [ ]:
print('Survived people with boat number %', round(df.loc[df.boat.notna(),'survived'].mean(),4)*100)
print('Survived people without boat number %', round(df.loc[df.boat.isna(),'survived'].mean(),4)*100)

In [ ]:
print('Not Survived people with boat number %', round(df.loc[df.body.notna(),'survived'].mean(),4)*100)

In [ ]:
df['boat'].unique()

In [ ]:
# Assignment = boat_assigned = 1 (who is survived), boat_assigned = 0 (who has not survived)

In [ ]:
df2 = df1.drop(['boat', 'home.dest'], axis = 1)

In [ ]:
df2

In [ ]:
df2.info()

### Annotation — Impute age with the mode (early exploratory path)
The mode is the most frequent age value, not the median. For a continuous variable such as age, median imputation is often more robust. This early manual cleaning occurs before the train/test split, so using it as the final modelling path can leak information from the test set; the later pipeline-based approach avoids that issue by fitting preprocessing on training data.

In [ ]:
modes = df['age'].mode().iloc[0]
df2['age']= df2['age'].fillna(modes)

In [ ]:
df2.info()

In [ ]:
print(type(df2['age'].mode()))

In [ ]:
print(modes)

### Annotation — Potential imputation bug
This cell computes the mode of `age` and uses it to fill missing `embarked` values. Those are different variables, so this is likely a mistake. For categorical `embarked`, use its own most-frequent value—or, preferably, the later categorical pipeline's `SimpleImputer(strategy='most_frequent')`. This annotation flags the issue; the original code is left unchanged.

In [ ]:
modes = df['age'].mode().iloc[0]
df2['embarked'] = df2['embarked'].fillna(modes)

### Annotation — Potential imputation bug
This cell again uses the mode of `age` to fill missing `fare`. That mixes variables and is likely incorrect. `fare` is numeric; the later numeric pipeline uses median imputation, which is generally a more defensible default. The original code is left unchanged.

In [ ]:
modes = df['age'].mode().iloc[0]
df2['fare'] = df2['fare'].fillna(modes)

In [ ]:
# median
# mode
#mean


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
plt.bar(df2['pclass'].unique(), df2['pclass'].value_counts())

In [ ]:
df2['pclass'].value_counts()

In [ ]:
a = plt.bar(df2['pclass'].unique(), df2['pclass'].value_counts().sort_index())
plt.bar_label(a)
plt.xlabel('pclass')
plt.ylabel('No of people')
plt.title('Class distribution of passenger in titanic')

### Annotation — Passenger class by sex
A count plot shows the number of passengers in each class, split by sex. Counts describe sample composition; they do not by themselves establish a causal relationship with survival.

In [ ]:
# How many male and female in class
ax = sns.countplot(data=df2,x='pclass',hue='sex')
for x in ax.containers:
    ax.bar_label(x)

plt.show()

In [ ]:
# How many of the survived are male and female
ay = sns.countplot(data=df2,x='survived',hue='sex')
for x in ay.containers:
    ay.bar_label(x)

plt.show()

In [ ]:
df2.info()

### Annotation — Correlation heatmap
The correlation matrix covers selected numeric columns. Pearson correlation captures linear association and may be misleading for ordinal/categorical codes such as `pclass`; it also does not establish causation. The heatmap is exploratory, not a feature-selection rule on its own.

In [ ]:
az = df2[['pclass','survived','age','sibsp','parch','fare']].corr()
sns.heatmap(az)

In [ ]:
df2['pclass'].unique()

In [ ]:
df2['sibsp'].unique()

In [ ]:
df2['parch'].unique()

In [ ]:
# how many poeple survived in sibsp
# am = df2[['sibsp','parch']]
sns.histplot(data=df2, x='sibsp',bins=8, kde=True, hue='survived')

In [ ]:
sns_values = sns.countplot(data= df2, x= 'sibsp', hue= 'survived')
for val in sns_values.containers:
  sns_values.bar_label(val)

In [ ]:
from sklearn.model_selection import train_test_split

### Annotation — Separate target and predictors
`survived` is the target `y`; all remaining columns become predictors `x`. Confirm that no predictor directly reveals the outcome or would be unavailable at inference time. In particular, boat/body-related fields can cause target leakage.

In [ ]:
y = df2['survived']
x = df2.drop(columns=['survived'])

In [ ]:
y.head()

In [ ]:
x.head()

### Annotation — Stratified train/test split
Reserves 20% for testing and uses `stratify=y` to preserve the target-class proportion in both splits. `random_state=5` makes the split reproducible. Split before fitting any imputer, scaler, encoder, or feature-selection step.

In [ ]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2, random_state=5,stratify=y)

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
num_features = ['age','fare','sibsp','parch'] # (0-1) numerical feature = how, how many 
cat_features = ['pclass','sex','ticket','embarked'] # categorical feature = which , which class what type - (understand only on numbers) other way of scaling called encoding [encoding means change the categories to number] 

In [ ]:
df2['embarked'].unique()

In [ ]:
# imputation ?


### Annotation — Define the modelling dataset
This drops `boat` and `home.dest` before building the model table. Dropping `boat` is important because it can reveal survival/evacuation outcome. The `errors='ignore'` option makes the cell tolerant of absent columns, but can also conceal a naming mistake.

In [ ]:
# we are cleaning our df1 df further to remove missing values in our boat and home.dest columns
model_df = df1.drop(columns=['boat','home.dest'],errors='ignore').copy()

In [ ]:
model_df

In [ ]:
num_features = ['age','fare','sibsp','parch']
cat_features = ['pclass','sex','ticket','embarked']

In [ ]:
selected_features = num_features + cat_features

### Annotation — Select X and y
`X` contains the chosen input features; `y` contains the binary survival target as integers. The feature list is explicit, which makes it easier to audit which information the model receives.

In [ ]:
# mapping input and outputs of the model 
X = model_df[selected_features].copy()
y = model_df['survived'].astype(int).copy()

In [ ]:
print("Selected input features ::")
print(selected_features)

In [ ]:
print("Shape of X : ",X.shape)
print("Shape of y : ", y.shape)

In [ ]:
display(X.head())

In [ ]:
# why we didn't used name directly ?


### Annotation — Split before preprocessing
This split is the one used by the later pipeline workflow. The preprocessing objects are fit inside the scikit-learn pipeline, so imputation, scaling, and category discovery are learned from the training folds rather than the full dataset.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2, # considering test data would be 20% of the model_df 
    random_state = 5,
    stratify=y # to preserve the survival-class proportion 
)

In [ ]:
print("Training Feature Dataset Shape: ",X_train.shape)
print("Training Target Dataset Shape", y_train.shape)
print("Testing Feature Dataset Shape: ",X_test.shape)
print("Testing Target Dataset Shape", y_test.shape)

In [ ]:
print("Training class proportion")
display(y_train.value_counts(normalize=True).round(3))


In [ ]:
print("Test class proportions")
display(y_test.value_counts(normalize=True).round(3))

### Annotation — Why use a pipeline?
A pipeline executes preprocessing and modelling steps in a fixed order. It reduces the risk of applying transformations inconsistently at training and prediction time. When used with cross-validation, preprocessing is refit within each training fold, reducing data leakage.

# What is pipeline ? 
- A ML workflow usually contains several steps in an ordered manner. 
- Ex piepline : 
 Raw Data -> Missing Value Processing/Treatment -> Encoding/Scaling -> Model Training
- sklearn has Pipeline object which has those features 


## Numerical Processing

In [ ]:
# Numerical processing 
num_features

### Annotation — Numeric preprocessing
The numeric pipeline first fills missing numeric values with the training-set median, then standardizes features. Standardization subtracts the training mean and divides by the training standard deviation; it is useful for logistic regression because feature scales affect optimization and regularization. The fitted statistics are learned only during `fit`.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

numeric_pipeline = Pipeline(steps=[
    ('imputer',SimpleImputer(strategy='median')),
    ('scaler',StandardScaler())
]    
)
numeric_pipeline

In [ ]:
# create similar pipeline for categorical features 
['pclass','sex','ticket','embarked']

In [ ]:
# SimpleImputer(strategy='most_frequent')
# OneHotEncoder

#sex = Male(01) , Female(10)
#handle_unknown = 'ignore'

### Annotation — Categorical preprocessing
The categorical pipeline fills missing categories with the most frequent category, then one-hot encodes each category into indicator columns. `handle_unknown='ignore'` prevents prediction from failing if a category appears at inference time but was absent during fitting; an unseen category is represented by zeros across that feature's learned indicator columns.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

categorical_pipeline = Pipeline(steps=[
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

categorical_pipeline

### Annotation — Apply different transformations by column
`ColumnTransformer` routes numeric columns through `numeric_pipeline` and categorical columns through `categorical_pipeline`. `remainder='drop'` discards any columns not explicitly listed, so the feature lists must include every intended predictor.

In [ ]:
# ColumnTransformer
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ('num',numeric_pipeline, num_features),
        ('cat',categorical_pipeline, cat_features)
    ],
    remainder='drop'
)

preprocessor

### Annotation — End-to-end logistic-regression pipeline
The outer pipeline applies the combined preprocessor and then fits logistic regression. Calling `.fit(X_train, y_train)` fits the imputers, scaler, encoder, and classifier in sequence. At prediction time, the same fitted transformations are applied automatically.

In [ ]:
# Logistic Regression Model 
from sklearn.linear_model import LogisticRegression

logreg_pipeline = Pipeline(steps=[
    ('preprocessor',preprocessor),
    ('classifier', LogisticRegression(max_iter=2000,random_state=5))
])

logreg_pipeline


### Annotation — Baseline evaluation
Training and test accuracy provide a first check for overfitting, but accuracy alone can hide poor performance on a minority class. Compare the gap between training and test scores and inspect precision, recall, F1, and the confusion matrix below.

In [ ]:
# create a baseline 
logreg_pipeline.fit(X_train, y_train)

train_accuracy = logreg_pipeline.score(X_train,y_train)
test_accuracy = logreg_pipeline.score(X_test, y_test)

print(f"Training accuracy:: ",train_accuracy)
print(f"Test accuracy :: ", test_accuracy)

In [ ]:
y_pred = logreg_pipeline.predict(X_test)
y_prob = logreg_pipeline.predict_proba(X_test)[:,1]

In [ ]:
prediction_demo = pd.DataFrame({
    'Actual': y_test.iloc[:10].values,
    'Predicted' : y_pred[:10],
    'P(Survived)': y_prob[0:10]
})

display(prediction_demo.round(4))

### model evaluation 
 - accuracy  ( TP + TN / TP + TN + FP + FN)
 - precision ( TP / TP + FP)
 - recall ( TP / TP + FN)
 - F1-score (s * (Precision * Recall / Precision + Recall))

### confusion matrix 
- For survived = 1:
    - TP : survived, our model also predicted survived
    - TN : did not survived, our model also predicted not survived
    - FP : did not survived, but our model predicted as survived
    - FN : survived, but our model predicted as not survived

### Annotation — Metrics and confusion matrix
Accuracy measures the fraction of all predictions that are correct. Precision measures how many predicted survivors actually survived; recall measures how many actual survivors were found; F1 is their harmonic mean. ROC-AUC evaluates ranking across thresholds using probabilities. Interpret metrics in the context of class balance and the costs of false positives versus false negatives.

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    
)
import matplotlib.pyplot as plt

print(f"Accuracy : {accuracy_score(y_test,y_pred):.4f}")
print(f"ROC-AUC : {roc_auc_score(y_test,y_prob):.4f}")

print("\n")
print("Classification Report:")
print(classification_report(y_test,y_pred))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=['Not Survived', 'Survived']
)

plt.title('Baseline Logistic Regression')
plt.show()

### Annotation — Stratified cross-validation
Five stratified folds preserve class proportions in each fold. `cross_val_score` clones and refits the full pipeline for each fold, so preprocessing is learned from each fold's training partition. The mean summarizes performance; the standard deviation indicates fold-to-fold variability.

In [ ]:
# Cross Validation 
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=5
)


cv_scores = cross_val_score(
    logreg_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1
)

print("Accuracy from each fold : ", cv_scores.round(4))
print(f"Mean CV accuracy: {cv_scores.mean():.4f}")
print(f"CV standard deviation: {cv_scores.std():.4f}")

### Annotation — Second, compact pipeline definition
This cell recreates the preprocessing/model pipeline for hyperparameter tuning. It uses `embarked` but omits `ticket`, unlike the earlier `cat_features` list; ensure that this difference is intentional. The cell also resets the classifier configuration, so later tuning applies to this `model_pipeline` object.

In [ ]:
#------------------------_#
       # Test
#-------------------------#
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

num_features = ['age', 'fare', 'sibsp', 'parch']
cat_features = ['pclass', 'sex', 'embarked']

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, num_features),
    ('cat', categorical_pipeline, cat_features)
])

model_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])
model_pipeline

### Annotation — Why regularize logistic regression?
Logistic regression learns coefficients for the transformed features. Very large coefficients can indicate an overly flexible fit, especially with sparse one-hot features. Regularization penalizes coefficient magnitude to improve generalization; it does not guarantee a perfect model.

In [ ]:
# Optimization of Model 
# what does LogisticRegression model learns here ?  Coefficients 
# In order order to create a perfect model we might end up having huge coefficients
# which would make the model prediction in accurate further 


### Annotation — The inverse relationship between C and regularization
In scikit-learn's logistic regression, `C` is the inverse regularization strength: smaller `C` means stronger regularization; larger `C` means weaker regularization. Tune it using cross-validation, not the held-out test set.

In [ ]:
# Coefficients (C) 
# smaller the C -> more regularization would be their in the ML model 
# larger the C -> weak regularization would be their in the ML model 

### Annotation — Manual C search
This loop evaluates several values of `C` using the same cross-validation strategy. The chosen value should be based on training-fold validation results; keep the test set untouched until the final evaluation.

In [ ]:
for c in [0.01, 0.1, 1, 10, 100]:

    model_pipeline.set_params(
        classifier__C=c
    )

    scores = cross_val_score(
        model_pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring='accuracy'
    )

    print(
        "C =", c,
        "CV =", round(scores.mean(), 4)
    )

### Annotation — Define the hyperparameter grid
`GridSearchCV` will compare combinations of solver, penalty, and `C`. **Code issue to check:** the solver string `'lbfg'` appears to be misspelled; scikit-learn's solver name is `'lbfgs'`. The grid may fail until this is corrected. This notebook annotation does not modify the original code.

In [ ]:
#GridSearchCV 
from sklearn.model_selection import GridSearchCV

param_grid = [
    {
        'classifier__solver' : ['liblinear'],
        'classifier__penalty':['l1','l2'],
        'classifier__C': [0.01, 0.1, 1, 10, 100]
    },
    {
        'classifier__solver' : ['lbfg'],
        'classifier__penalty':['l2'],
        'classifier__C': [0.01, 0.1, 1, 10, 100]
    }
    
]



### Annotation — Run GridSearchCV
The search fits each valid parameter combination across the cross-validation folds and selects the configuration with the best mean score under the specified metric (`accuracy`). Because `refit=True` by default, the best configuration is then refit on the full training set.

In [ ]:
grid_search = GridSearchCV(
    estimator=model_pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(X_train,y_train)

In [ ]:
print("Best parameters : ", grid_search.best_params_)

In [ ]:
print("Best CV score : ", grid_search.best_score_)

In [ ]:
print("Best Grid Result ", grid_search.cv_results_)

In [ ]:
# how to get the hyperparameters for a given model 
for key in model_pipeline.get_params().keys():
    if key.startswith('classifier'):
        print(key)



In [ ]:
# create a simple ml pipeline which predicts if a given 
# student pass his examp or not 


### Annotation — Inspect the search results
The table extracts each parameter combination's mean and standard deviation of validation score plus its rank. Prefer the best mean score while considering variability and model simplicity; small score differences may not be meaningful.

In [ ]:
import pandas as pd 

grid_results = pd.DataFrame(
    grid_search.cv_results_
)
grid_results

columns_to_show = [
    'param_classifier__C',
    'param_classifier__penalty',
    'param_classifier__solver',
    'mean_test_score',
    'std_test_score',
    'rank_test_score'
    
]

grid_results[columns_to_show].sort_values('rank_test_score').head(5)

### Annotation — Retrieve the best fitted estimator
`best_estimator_` is the refitted pipeline corresponding to the best grid-search configuration. Use it for the final held-out test evaluation, rather than choosing parameters based on test performance.

In [ ]:
best_model = grid_search.best_estimator_
print(best_model)

In [ ]:
best_model

### Annotation — Final test-set evaluation
This is the final evaluation of the selected pipeline on the held-out test set. Accuracy, precision, recall, and F1 are calculated from predictions; predicted probabilities are also available for threshold-based analysis. Do not use these test results to continue tuning the model.

In [ ]:
# Final evaluation on test data
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay
)

# prediction using the best pipeline which we got from GridSearchCV
y_predict_best = best_model.predict(X_test)

# Probability of survived = 1
y_probab_best = best_model.predict_proba(X_test)[:,1]

print("Final Test Results")


print(
    "Accuracy :",
    round(accuracy_score(y_test, y_predict_best), 4)
)

print(
    "Precision:",
    round(precision_score(y_test, y_predict_best), 4)
)

print(
    "Recall   :",
    round(recall_score(y_test, y_predict_best), 4)
)

print(
    "F1 Score :",
    round(f1_score(y_test, y_predict_best), 4)
)

In [ ]:
print(
    classification_report(
        y_test,
        y_predict_best,
        target_names = [
            "not survived",
            "survived"
        ]
    )
)

In [ ]:
import matplotlib.pyplot as plt

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_predict_best,
    display_labels = [
            "not survived",
            "survived"
        ]
)

plt.title("Best Logistic Regression Model")
plt.show()

In [ ]:
# Cross Validation : How good is this configuration 
# GridSearchCV : Which configuration should I choose ? 